# 📊 Phase 1: Data Quality Assessment & Cleaning Pipeline
**Project:** E-Commerce Sales Analytics Portfolio Project
**Objective:** Systematically audit raw e-commerce transaction data, profile data quality, engineer business metrics, and export Star Schema dimension/fact tables.

In [ ]:
import pandas as pd
import numpy as np
import os

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 1000)

RAW_PATH = os.path.join('..', 'data', 'raw', 'Sample_Superstore.csv')
CLEANED_DIR = os.path.join('..', 'data', 'cleaned')
os.makedirs(CLEANED_DIR, exist_ok=True)

df_raw = pd.read_csv(RAW_PATH, encoding='windows-1252')
print(f'Raw records loaded: {df_raw.shape[0]:,} rows, {df_raw.shape[1]} columns')
df_raw.head(3)

## 1. Data Quality & Completeness Profiling
- Missing value audit across all 21 columns
- Duplicate records and key integrity

In [ ]:
missing = pd.DataFrame({
    'Missing_Count': df_raw.isnull().sum(),
    'Missing_Pct': (df_raw.isnull().sum() / len(df_raw)) * 100
})
print('=== Missing Value Audit ===')
print(missing if missing['Missing_Count'].sum() > 0 else '[OK] Zero missing values found across all columns.')

print('\n=== Duplication & Uniqueness ===')
print(f'Full duplicate rows: {df_raw.duplicated().sum()}')
print(f'Unique Row IDs: {df_raw["Row ID"].nunique():,} / {len(df_raw):,}')
print(f'Unique Order IDs: {df_raw["Order ID"].nunique():,}')
print(f'Order ID + Product ID multi-item line occurrences: {df_raw.duplicated(subset=["Order ID", "Product ID"]).sum()}')

## 2. Business Constraint Auditing & Outliers
- Temporal validity (`Ship Date` >= `Order Date`)
- Bounded domain checks (`Sales` > 0, `Quantity` >= 1, `Discount` in [0, 0.8])
- Profitability analysis (identifying loss-making transactions)

In [ ]:
order_dates = pd.to_datetime(df_raw['Order Date'], format='mixed')
ship_dates = pd.to_datetime(df_raw['Ship Date'], format='mixed')

invalid_shipping = (ship_dates < order_dates).sum()
print(f'Invalid shipping dates: {invalid_shipping}')
print(f'Non-positive Sales: {(df_raw["Sales"] <= 0).sum()}')
print(f'Non-positive Quantity: {(df_raw["Quantity"] <= 0).sum()}')
print(f'Discount bounds: min={df_raw["Discount"].min()}, max={df_raw["Discount"].max()}')

loss_orders = (df_raw['Profit'] < 0).sum()
print(f'Loss-Making Transactions: {loss_orders:,} ({loss_orders / len(df_raw):.2%})')

## 3. Data Cleaning, Transformation & Feature Engineering

In [ ]:
df_clean = df_raw.copy()
df_clean.columns = [c.strip().lower().replace(' ', '_').replace('-', '_') for c in df_clean.columns]

df_clean['order_date'] = pd.to_datetime(df_clean['order_date'], format='mixed')
df_clean['ship_date'] = pd.to_datetime(df_clean['ship_date'], format='mixed')
df_clean['postal_code'] = df_clean['postal_code'].astype(str).str.zfill(5)

# Temporal features
df_clean['shipping_duration_days'] = (df_clean['ship_date'] - df_clean['order_date']).dt.days
df_clean['order_year'] = df_clean['order_date'].dt.year
df_clean['order_month'] = df_clean['order_date'].dt.month
df_clean['order_month_name'] = df_clean['order_date'].dt.strftime('%B')
df_clean['order_year_month'] = df_clean['order_date'].dt.strftime('%Y-%m')
df_clean['order_quarter'] = df_clean['order_date'].dt.to_period('Q').astype(str)
df_clean['order_day_of_week'] = df_clean['order_date'].dt.day_name()

# Unit Economics
df_clean['unit_price'] = (df_clean['sales'] / df_clean['quantity']).round(2)
df_clean['unit_cost'] = ((df_clean['sales'] - df_clean['profit']) / df_clean['quantity']).round(2)
df_clean['profit_margin_pct'] = ((df_clean['profit'] / df_clean['sales']) * 100).round(2)
df_clean['is_profitable'] = (df_clean['profit'] > 0).astype(int)

# Discount brackets
def categorize_discount(d):
    if d == 0.0:
        return 'No Discount (0%)'
    elif d <= 0.20:
        return 'Low Discount (1-20%)'
    elif d <= 0.50:
        return 'Medium Discount (21-50%)'
    else:
        return 'High Discount (>50%)'

df_clean['discount_bracket'] = df_clean['discount'].apply(categorize_discount)

master_path = os.path.join(CLEANED_DIR, 'superstore_cleaned.csv')
df_clean.to_csv(master_path, index=False)
print(f'Master Cleaned Dataset Saved: {df_clean.shape}')

## 4. Star Schema Dimension & Fact Modeling
Generates `dim_customers`, `dim_products`, `dim_geography`, `dim_dates`, and `fact_sales`.

In [ ]:
dim_customers = df_clean[['customer_id', 'customer_name', 'segment']].drop_duplicates(subset=['customer_id']).copy()
dim_customers.to_csv(os.path.join(CLEANED_DIR, 'dim_customers.csv'), index=False)

dim_products = df_clean[['product_id', 'category', 'sub_category', 'product_name']].drop_duplicates(subset=['product_id']).copy()
dim_products.to_csv(os.path.join(CLEANED_DIR, 'dim_products.csv'), index=False)

dim_geography = df_clean[['postal_code', 'city', 'state', 'region', 'country']].drop_duplicates().copy()
dim_geography.to_csv(os.path.join(CLEANED_DIR, 'dim_geography.csv'), index=False)

min_date = df_clean['order_date'].min()
max_date = max(df_clean['order_date'].max(), df_clean['ship_date'].max())
dim_dates = pd.DataFrame({'date': pd.date_range(start=min_date, end=max_date, freq='D')})
dim_dates['date_id'] = dim_dates['date'].dt.strftime('%Y%m%d').astype(int)
dim_dates['year'] = dim_dates['date'].dt.year
dim_dates['quarter'] = dim_dates['date'].dt.to_period('Q').astype(str)
dim_dates['month'] = dim_dates['date'].dt.month
dim_dates['month_name'] = dim_dates['date'].dt.strftime('%B')
dim_dates['day'] = dim_dates['date'].dt.day
dim_dates['day_name'] = dim_dates['date'].dt.strftime('%A')
dim_dates['is_weekend'] = dim_dates['date'].dt.dayofweek.isin([5, 6]).astype(int)
dim_dates.to_csv(os.path.join(CLEANED_DIR, 'dim_dates.csv'), index=False)

fact_cols = [
    'row_id', 'order_id', 'order_date', 'ship_date', 'ship_mode',
    'customer_id', 'postal_code', 'product_id',
    'sales', 'quantity', 'discount', 'profit',
    'shipping_duration_days', 'unit_price', 'unit_cost',
    'profit_margin_pct', 'is_profitable', 'discount_bracket'
]
fact_sales = df_clean[fact_cols].copy()
fact_sales.to_csv(os.path.join(CLEANED_DIR, 'fact_sales.csv'), index=False)

print('Dimension and Fact tables successfully exported.')